# Corpus overview (n = 25)

Exploratory **data-quality** notebook for the live BoardGameGeek smoke corpus.

This is **not** a predictive analysis. It does not infer fun, quality,
popularity, or what causes ratings or complexity. BGG mechanic and category
labels are **source facts**, not this project's future ontology.

Sample size is **25 games**. Findings here describe this sample only.

| Kind | Meaning |
| --- | --- |
| Source fact | A value stored on `Game` as reported by BGG |
| Data-quality observation | Completeness, duplicates, ranges, anomalies |
| Hypothesis | A question for a larger corpus, not a claim |


In [ ]:
from __future__ import annotations

from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from board_game_analysis.ingestion.quality import (
    category_counts,
    flag_anomalies,
    integrity_report,
    list_length_summaries,
    load_games_jsonl,
    load_manifest,
    mechanic_counts,
    missingness_rows,
    numeric_summaries,
)


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "board_game_analysis"
        ).is_dir():
            return candidate
    raise FileNotFoundError("could not locate repository root")


def table(headers: list[str], rows: list[list[object]]) -> None:
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    for row in rows:
        lines.append("| " + " | ".join(str(cell) for cell in row) + " |")
    display(Markdown("\n".join(lines)))


ROOT = repo_root()
JSONL = ROOT / "data" / "processed" / "boardgamegeek" / "corpus_v0.jsonl"
MANIFEST = ROOT / "data" / "derived" / "corpus" / "bgg_boardgames_v0.manifest.json"
RAW_DIR = ROOT / "data" / "raw" / "boardgamegeek"
CURRENT_YEAR = datetime.now(UTC).year

print(f"repo root: {ROOT}")
print(f"jsonl:     {JSONL}")
print(f"manifest:  {MANIFEST}")
print(f"exists:    jsonl={JSONL.is_file()} manifest={MANIFEST.is_file()}")

## 1. Load and validate

Records are loaded from the processed JSONL and validated with the canonical
Pydantic `Game` model. Invalid lines are **kept and listed**, not dropped.


In [ ]:
if not JSONL.is_file():
    raise FileNotFoundError(
        f"missing {JSONL}. Run: uv run board-game-ingest --corpus --limit 25"
    )

loaded = load_games_jsonl(JSONL)
manifest = load_manifest(MANIFEST) if MANIFEST.is_file() else None
games = loaded.games
n = len(games)

display(Markdown(f"**Validated `Game` records:** {n}"))
if loaded.errors:
    display(Markdown("**Load / validation errors (not discarded):**"))
    table(
        ["line", "id", "reason"],
        [
            [err.line_number, err.raw_id or "", err.reason.replace("\n", " ")]
            for err in loaded.errors
        ],
    )
else:
    display(Markdown("Every JSONL line validated as `Game`."))

## 2. Corpus integrity

**Source facts:** the smoke run requested 25 BGG thing ids and wrote one
JSONL line per ingested board game.


In [ ]:
report = integrity_report(loaded, manifest)
integrity_rows = [
    ["JSONL records (objects)", report.n_jsonl_records],
    ["Validated Game records", n],
    ["Unique game IDs", report.n_unique_ids],
    ["Unique titles", report.n_unique_titles],
    ["Duplicate IDs", ", ".join(report.duplicate_ids) or "none"],
    ["Duplicate titles", ", ".join(report.duplicate_titles) or "none"],
    ["Manifest requested", report.n_manifest_requested],
    ["Manifest ok", report.n_manifest_ok],
    ["Manifest skipped", report.n_manifest_skipped],
    ["Manifest errors", report.n_manifest_errors],
    [
        "JSONL ids missing from manifest",
        ", ".join(report.jsonl_ids_missing_from_manifest) or "none",
    ],
    [
        "Manifest ok ids missing from JSONL",
        ", ".join(report.manifest_ok_ids_missing_from_jsonl) or "none",
    ],
    ["Manifest item types", report.manifest_item_types or "n/a"],
    ["Provenance issues", len(report.provenance_issues)],
    [
        "Title heuristic: expansion-like",
        ", ".join(report.expansion_like_titles) or "none",
    ],
]
table(["Check", "Value"], integrity_rows)

if report.provenance_issues:
    display(Markdown("**Provenance issues:**"))
    for issue in report.provenance_issues:
        display(Markdown(f"- {issue}"))
else:
    display(
        Markdown(
            "Every validated game has one `SourceReference` with "
            "`source=boardgamegeek`, `source_type=xmlapi2`, a `/thing` URL, "
            "a source identifier that matches `Game.id`, and `retrieved_at`."
        )
    )

In [ ]:
table(
    ["id", "title", "year", "players", "time (min)", "rating", "n ratings", "weight"],
    [
        [
            game.id,
            game.title,
            game.release_year,
            f"{game.min_players}–{game.max_players}",
            f"{game.min_play_time_minutes}–{game.max_play_time_minutes}",
            game.rating,
            game.rating_count,
            game.complexity,
        ]
        for game in games
    ],
)

### Integrity observations

- **Observation:** 25 JSONL records, 25 unique ids, 25 unique titles, 25
  manifest `ok` rows, 0 skipped, 0 errors.
- **Observation:** manifest `item_type` is `boardgame` for all 25.
- **Observation:** titles such as *Pandemic Legacy: Season 1* and
  *7 Wonders Duel* are standalone BGG `boardgame` items, not expansions.
  A colon in the title is not an expansion signal.
- **Hypothesis:** a 500/1,000-id list will include true expansions if ids
  are not curated; the item-type filter is what must catch them.


## 3. Missingness

List fields report **null**, **empty list**, and **non-empty list**
separately. After `Game` validation, lists cannot be JSON `null` (the model
uses `list[...]` with a default of `[]`). A null list in the JSONL would
appear above as a validation error.


In [ ]:
rows = missingness_rows(games)
table(
    ["field", "kind", "present / non-empty", "null", "% null", "empty list", "% empty"],
    [
        [
            row.field,
            row.kind,
            row.present,
            row.missing_null,
            f"{row.pct_missing_null:.1f}",
            row.empty_list if row.empty_list is not None else "—",
            f"{row.pct_empty_list:.1f}" if row.pct_empty_list is not None else "—",
        ]
        for row in rows
    ],
)

### Missingness observations

- **Observation:** `popularity` is null for **25/25** (100%). That matches
  the v0 contract: BGG rank and `owned` are not mapped.
- **Observation:** every other `Game` field is present. No empty designer,
  publisher, category, mechanic, or source lists in this sample.
- **Hypothesis:** this 25-game list is famous, heavily documented titles.
  It does **not** test the missing-value policy (`0` → null, unrated games,
  `rating_count=0`). A larger corpus must include obscure and unrated items
  before we treat that policy as empirically confirmed on live data.


## 4. Numeric distributions

`rating_count` is plotted on a log scale so a few high-count games do not
flatten the rest. This sample is still a high-count slice of BGG.


In [ ]:
summaries = numeric_summaries(games)
table(
    ["field", "n", "min", "median", "mean", "max", "stdev"],
    [
        [
            row.field,
            row.count,
            "—" if row.minimum is None else round(row.minimum, 4),
            "—" if row.median is None else round(row.median, 4),
            "—" if row.mean is None else round(row.mean, 4),
            "—" if row.maximum is None else round(row.maximum, 4),
            "—" if row.stdev is None else round(row.stdev, 4),
        ]
        for row in summaries
    ],
)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 5.5))
fig.suptitle(f"Numeric distributions (n={n})", fontsize=12)


def hist(ax, values, title, *, bins=10, log_x=False):
    ax.hist(values, bins=bins, color="#4C6A92", edgecolor="white")
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("games")
    if log_x:
        ax.set_xscale("log")
        ax.set_xlabel("log scale")


hist(axes[0, 0], [g.release_year for g in games], "release_year")
hist(
    axes[0, 1],
    [g.min_players for g in games],
    "min_players",
    bins=range(1, 6),
)
hist(
    axes[0, 2],
    [g.max_players for g in games],
    "max_players",
    bins=range(2, 10),
)
hist(axes[0, 3], [g.min_play_time_minutes for g in games], "min play time")
hist(axes[1, 0], [g.max_play_time_minutes for g in games], "max play time")
hist(axes[1, 1], [g.rating for g in games], "rating (1–10)")
hist(
    axes[1, 2],
    [g.rating_count for g in games],
    "rating_count (log x)",
    log_x=True,
)
hist(axes[1, 3], [g.complexity for g in games], "complexity (1–5)")
fig.tight_layout()
plt.show()

### Distribution observations

- **Observation:** years run 1995–2019 (median 2014). This smoke list is
  modern hobby-game hits, not a historical sample.
- **Observation:** ratings sit in a narrow high band (~7.03–8.56). That is
  selection, not a BGG-wide rating distribution.
- **Observation:** every game has tens of thousands of ratings (min 33,619).
  `rating_count` is right-skewed even inside this famous-game slice.
- **Observation:** complexity spans ~1.25 (Codenames) to ~4.08 (Spirit
  Island), which is a usable spread for a later structural comparison —
  still not a claim about difficulty causing ratings.
- **Hypothesis:** a 500/1,000 corpus will have lower medians for rating and
  rating_count, and will include years before 1995 and after 2019.


## 5. Structural fields (source labels)

Designer / publisher / category / mechanic **counts** are source-list
lengths. Mechanic names below are BGG labels, not a project taxonomy.


In [ ]:
length_rows = list_length_summaries(games)
table(
    ["field", "n", "min", "median", "mean", "max"],
    [
        [
            row.field,
            row.count,
            row.minimum,
            round(row.median, 2),
            round(row.mean, 2),
            row.maximum,
        ]
        for row in length_rows
    ],
)

table(
    ["title", "designers", "publishers", "categories", "mechanics"],
    [
        [
            game.title,
            len(game.designers),
            len(game.publishers),
            len(game.categories),
            len(game.mechanics),
        ]
        for game in games
    ],
)

In [ ]:
cats = category_counts(games)
mechs = mechanic_counts(games)
display(Markdown(f"**Distinct BGG categories in sample:** {len(cats)}"))
table(["category (BGG label)", "games"], [list(item) for item in cats[:12]])
display(Markdown(f"**Distinct BGG mechanics in sample:** {len(mechs)}"))
table(["mechanic (BGG label)", "games"], [list(item) for item in mechs[:15]])

### Structural observations

- **Observation:** designers are usually 1 (max 3). Publishers are many:
  min 11, median 21, max 52 (Catan). That is BGG listing every regional
  publisher, not a normalization error.
- **Observation:** mechanic lists are long (median 11, Gloomhaven 27).
  Live Catan has 15 mechanic links; the older test fixture only had 2.
  BGG has been adding tags.
- **Observation:** most common categories here are Economic and Card Game.
  Most common mechanic labels are Hand Management and Variable Set-up.
  These are source frequencies, not design conclusions.


## 6. Suspicious / extreme values

Flags are **heuristics for review**, not rejection rules and not schema
changes. Large publisher/mechanic lists are expected for famous BGG titles.


In [ ]:
flags = flag_anomalies(games, current_year=CURRENT_YEAR)
display(Markdown(f"**Flags raised:** {len(flags)} (n={n})"))
if flags:
    table(
        ["id", "title", "code", "detail"],
        [[f.game_id, f.title, f.code, f.detail] for f in flags],
    )
else:
    display(Markdown("No heuristic flags."))

equal_time = [
    game.title
    for game in games
    if game.min_play_time_minutes == game.max_play_time_minutes
]
display(
    Markdown(
        f"**Equal min/max play time ({len(equal_time)}/{n}):** " + ", ".join(equal_time)
    )
)

### Anomaly observations

- **Observation:** no inverted player or time ranges, no off-scale
  rating/complexity, no negative/zero player or time values, no
  rating-without-count inconsistency.
- **Observation:** several games flag `large_publisher_list` or
  `large_mechanic_list`. Those match raw BGG link counts. Do not trim them.
- **Observation:** equal min/max play time is common (BGG often stores a
  single `playingtime` on both bounds). That is valid, not inverted.
- **Observation:** 2-player-only games (Twilight Struggle, 7 Wonders Duel)
  have `min_players == max_players == 2`. Allowed by the model.


## 7. Semantic checks against live BGG XML

The raw archive is read **read-only**. Nothing here rewrites `data/raw/`.


In [ ]:
import xml.etree.ElementTree as ET

raw_files = sorted(path for path in RAW_DIR.glob("*.xml") if path.stem.isdigit())
display(Markdown(f"**Raw per-id XML files:** {len(raw_files)}"))

item_types: Counter[str] = Counter()
has_rank = has_owned = has_bayes = 0
usersrated_zero = 0
numweights_zero = 0
year_zero = 0
expansion_links = 0
for path in raw_files:
    item = ET.fromstring(path.read_text(encoding="utf-8")).find("item")
    if item is None:
        continue
    item_types[item.attrib.get("type", "")] += 1
    year = item.find("yearpublished")
    if year is not None and year.attrib.get("value") == "0":
        year_zero += 1
    stats = item.find("statistics/ratings")
    if stats is None:
        continue
    ur = stats.find("usersrated")
    nw = stats.find("numweights")
    if ur is not None and ur.attrib.get("value") == "0":
        usersrated_zero += 1
    if nw is not None and nw.attrib.get("value") == "0":
        numweights_zero += 1
    if stats.find("ranks") is not None:
        has_rank += 1
    if stats.find("owned") is not None:
        has_owned += 1
    if stats.find("bayesaverage") is not None:
        has_bayes += 1
    expansion_links += sum(
        1
        for link in item.findall("link")
        if link.attrib.get("type") == "boardgameexpansion"
    )

table(
    ["raw XML check", "value"],
    [
        ["item types", dict(item_types)],
        ["yearpublished=0", year_zero],
        ["usersrated=0", usersrated_zero],
        ["numweights=0", numweights_zero],
        ["has ranks", f"{has_rank}/{len(raw_files)}"],
        ["has owned", f"{has_owned}/{len(raw_files)}"],
        ["has bayesaverage", f"{has_bayes}/{len(raw_files)}"],
        ["boardgameexpansion links (sum)", expansion_links],
        ["popularity non-null in Game", sum(g.popularity is not None for g in games)],
        ["rating_count == 0 in Game", sum(g.rating_count == 0 for g in games)],
    ],
)

catan_raw = RAW_DIR / "13.xml"
if catan_raw.is_file():
    item = ET.fromstring(catan_raw.read_text(encoding="utf-8")).find("item")
    stats = item.find("statistics/ratings")
    rank = stats.find("ranks/rank")
    display(
        Markdown(
            "Catan raw vs processed (example of unused BGG signals): "
            f"average={stats.find('average').attrib.get('value')}, "
            f"bayesaverage={stats.find('bayesaverage').attrib.get('value')}, "
            f"owned={stats.find('owned').attrib.get('value')}, "
            f"boardgame rank="
            f"{rank.attrib.get('value') if rank is not None else 'n/a'}. "
            f"Processed popularity="
            f"{next(g.popularity for g in games if g.id == 'bgg-13')}."
        )
    )

### Semantic conclusions (this sample)

| Assumption | Live 25-game evidence |
| --- | --- |
| Missing year/players/time as `0` | **Not observed.** Every raw file has non-zero values. Policy still covered by fixtures, not by this sample. |
| `rating_count=0` | **Not observed.** Minimum is 33,619. |
| Placeholder rating/weight when unvoted | **Not observed.** Every item has `usersrated` and `numweights` > 0. |
| Publisher multiplicity | **Confirmed.** 11–52 names; Catan raw has 52 `boardgamepublisher` links. |
| Category/mechanic multiplicity | **Confirmed.** Categories 1–7; mechanics 4–27. |
| Player-count edges | Solo-capable games (`min_players=1`) and 2-player-only games occur. No inversions. |
| Play-time edges | Equal min/max is common. Wide ranges (Agricola 30–150) occur. |
| `popularity` stays null | **Confirmed.** All 25 raw files have `ranks` and `owned`; none were mapped. Catan rank ~627 and owned ~245k stay unused. |
| Expansion/accessory filter | **Appears to work.** All 25 item types are `boardgame`. Base games can have dozens of expansion *links* (Catan has 100+) without being skipped. |

No normalizer or schema change is justified from this sample. The missing-value
rules remain fixture-tested, not live-tested.


## 8. Ready to scale? (hypotheses, not a go/no-go from n=25)

**Observations that support scaling the *pipeline*:**

- 25/25 ingested, validated, and provenance-complete
- batching and resume already used (Catan cached; others fetched in batches)
- item-type filter did not drop standalone games that have expansion links
- leaving `popularity` null avoids mixing rank/ownership into `Game`

**Observations that limit what this sample can certify:**

- no missing scalars except the intentional `popularity` hole
- no unrated or zero-weight games
- no expansions in the requested id list
- rating/rating_count distributions are a bestseller slice

**Hypotheses for a 500/1,000 run:**

1. Missingness will appear; keep treating BGG `0` as unknown except
   `rating_count`.
2. Some requested ids will be expansions/accessories; expect `skipped` > 0.
3. Publisher and mechanic lists will stay long for hits and stay short
   (or empty) for obscure titles — both are valid source facts.
4. Do not derive popularity from rank or `owned` without a new, explicit
   field and contract.

Next scale-up should keep the frozen id list curated, keep raw XML
immutable, and rerun this notebook rather than inventing corrections.
